# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("License:", dataset.metadata.license)
print("Published Date:", dataset.metadata.datePublished)


## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset contains survey data and consists of multiple record sets. Let's inspect them using their `@id` values.

In [ ]:
# List all record set @id's and their fields.
record_sets = []

for rs in dataset.metadata.recordSet:
    record_sets.append(rs['@id'])
    print(f"RecordSet @id: {rs['@id']}")
    print("Fields:")
    if 'field' in rs:
        for fld in rs['field']:
            print(f"  - Field @id: {fld['@id']} (name: {fld.get('name','')})")
    print('-'*40)

# Show sample records from each record set
for rs_id in record_sets:
    print(f"Records from RecordSet @id: {rs_id}")
    for x in dataset.records(record_set=rs_id):
        print(x)
        break  # Print only the first example from each set to avoid flooding output
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Let's extract all record sets for exploration.

In [ ]:
# Extract data from each record set
# record_sets is already fetched from previous cell
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"DataFrame columns for RecordSet @id: {record_set}:\n{df.columns.tolist()}")
        print(df.head())
        print('-'*40)


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, let's assume the main survey record set has a column for GAD-7 score (often used as a numeric indicator for anxiety). We'll reference by the proper `@id` if available. Adjust the variable names below to use real column names (`@id`) from output above.

In [ ]:
# Example: EDA on GAD-7 numeric scores
# Adjust the '@id's as found in previous cells

# Suppose the primary record set is called 'cr:RecordSet_survey' (update with real @id!)
primary_rs_id = record_sets[0] if record_sets else None
df = dataframes.get(primary_rs_id, pd.DataFrame())

if not df.empty:
    # Find numeric fields (e.g., GAD-7, PHQ-9, PSQ)
    numeric_fields = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower()]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = 10
        # Remove rows with missing/invalid values
        valid = df[numeric_field].apply(lambda x: isinstance(x, (int, float)))
        filtered_df = df[valid & (df[numeric_field] > threshold)]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by demographic field, e.g., 'gender' or similar
        group_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'age' in col.lower()]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print("No survey records loaded or record set unavailable.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's visualize the distribution of the main numeric indicator (e.g., GAD-7) and compare groups if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_fields:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field], kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field], palette='pastel')
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey dataset provides valuable insights into psychological symptom distributions and demographic correlations in Kilifi County, Kenya.
- Using `mlcroissant`, we accessed metadata, record sets, and performed preliminary EDA and visualization.
- Initial analysis shows variability in mental health scores, potentially correlated with demographic factors.
- The standardized Croissant format makes the data AI-ready and easy to manipulate for community health initiatives and research.

> Please adjust field names and `@id`s in this notebook with precise values as discovered during exploration for your particular use case.